In [2]:
# ============================================================
# BLOCK 1: Environment Setup
# ============================================================
!pip install wfdb neurokit2 PyWavelets -q

import wfdb
import numpy as np
import pandas as pd
import os, warnings
from scipy.signal import butter, filtfilt, iirnotch, savgol_filter, welch
from scipy.interpolate import interp1d
from scipy.stats import skew, kurtosis
import neurokit2 as nk
import pywt
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for image generation
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
print("All imports done.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 4.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 688.9/688.9 kB 40.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 85.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
All imports done.


In [3]:
# ============================================================
# BLOCK 2: Load PhysioNet Apnea-ECG Data (same as ML project)
# ============================================================
record_ids = [
    'a01', 'a02', 'a03', 'a04', 'a05',   # heavy apnea
    'a10', 'a15', 'a20',                   # moderate apnea
    'b01', 'b02', 'b03', 'b04',            # borderline
    'c01', 'c02', 'c03',                   # normal-dominant
]

database = {}
for rid in record_ids:
    try:
        rec = wfdb.rdrecord(rid, pn_dir='apnea-ecg')
        ann = wfdb.rdann(rid, 'apn', pn_dir='apnea-ecg')
        signal = rec.p_signal[:, 0]
        fs = rec.fs
        labels = ann.symbol
        database[rid] = {
            'signal': signal, 'fs': fs,
            'labels': labels,
            'duration_hrs': len(signal) / fs / 3600
        }
        a_count = labels.count('A')
        n_count = labels.count('N')
        print(f"{rid}: {len(signal)/fs/60:.0f} min | A={a_count} N={n_count} | apnea_ratio={a_count/(a_count+n_count):.2f}")
    except Exception as e:
        print(f"{rid}: FAILED - {e}")

print(f"\nTotal records loaded: {len(database)}")
total_a = sum(d['labels'].count('A') for d in database.values())
total_n = sum(d['labels'].count('N') for d in database.values())
print(f"Overall: A={total_a} N={total_n} | ratio={total_a/(total_a+total_n):.2f}")

a01: 493 min | A=470 N=19 | apnea_ratio=0.96
a02: 530 min | A=420 N=108 | apnea_ratio=0.80
a03: 522 min | A=246 N=273 | apnea_ratio=0.47
a04: 497 min | A=453 N=39 | apnea_ratio=0.92
a05: 453 min | A=276 N=178 | apnea_ratio=0.61
a10: 517 min | A=100 N=417 | apnea_ratio=0.19
a15: 509 min | A=368 N=142 | apnea_ratio=0.72
a20: 510 min | A=315 N=195 | apnea_ratio=0.62
b01: 486 min | A=19 N=468 | apnea_ratio=0.04
b02: 528 min | A=93 N=424 | apnea_ratio=0.18
b03: 440 min | A=73 N=368 | apnea_ratio=0.17
b04: 429 min | A=10 N=419 | apnea_ratio=0.02
c01: 483 min | A=0 N=484 | apnea_ratio=0.00
c02: 501 min | A=1 N=501 | apnea_ratio=0.00
c03: 453 min | A=0 N=454 | apnea_ratio=0.00

Total records loaded: 15
Overall: A=2844 N=4489 | ratio=0.39


In [5]:
# ============================================================
# CELL 2: Imports
# ============================================================

!pip install biosppy -q
!pip install peakutils -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from scipy.signal import butter, filtfilt, welch
from biosppy.signals.ecg import hamilton_segmenter
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_curve, auc, accuracy_score, f1_score,
                             precision_score, recall_score)
print("All imports done.")

All imports done.


In [7]:
# ============================================================
# CELL 3: ECG Preprocessing & RR Interval Extraction
# ============================================================

def bandpass_filter(signal, fs, lowcut=0.5, highcut=40.0, order=4):
    nyq = 0.5 * fs
    b, a = butter(order, [lowcut/nyq, highcut/nyq], btype='band')
    return filtfilt(b, a, signal)

def get_rpeaks(signal, fs):
    """R-peak detection using Hamilton segmenter"""
    try:
        rpeaks = hamilton_segmenter(signal=signal, sampling_rate=fs)
        return rpeaks['rpeaks']
    except:
        return np.array([])

# Process all records: filter → R-peaks → RR intervals per minute
processed = {}

for rec_name, rec_data in database.items():
    signal = rec_data['signal']
    fs = rec_data['fs']
    labels = rec_data['labels']

    # Bandpass filter
    filtered = bandpass_filter(signal, fs)

    # R-peak detection on full signal
    rpeaks = get_rpeaks(filtered, fs)

    if len(rpeaks) < 2:
        print(f"{rec_name}: SKIPPED (no R-peaks found)")
        continue

    # Convert R-peak sample indices to seconds
    rpeak_times = rpeaks / fs

    # RR intervals in ms
    rr_intervals = np.diff(rpeak_times) * 1000  # ms
    rr_times = rpeak_times[1:]  # timestamp of each RR interval

    # Segment into 1-minute windows aligned with annotations
    samples_per_min = 60 * fs
    n_minutes = len(labels)

    minute_rr = []
    minute_labels = []

    for i in range(n_minutes):
        t_start = i * 60  # seconds
        t_end = (i + 1) * 60

        # Get RR intervals falling in this minute
        mask = (rr_times >= t_start) & (rr_times < t_end)
        rr_seg = rr_intervals[mask]

        if len(rr_seg) >= 5:  # need at least 5 beats for meaningful HRV
            minute_rr.append(rr_seg)
            minute_labels.append(1 if labels[i] == 'A' else 0)

    processed[rec_name] = {
        'rr_segments': minute_rr,
        'labels': minute_labels
    }

    n_a = sum(1 for l in minute_labels if l == 1)
    n_n = sum(1 for l in minute_labels if l == 0)
    print(f"{rec_name}: {len(minute_labels)} valid segments | A={n_a}, N={n_n}")

total_seg = sum(len(p['labels']) for p in processed.values())
print(f"\nTotal valid 1-min segments: {total_seg}")

a01: 489 valid segments | A=470, N=19
a02: 528 valid segments | A=420, N=108
a03: 519 valid segments | A=246, N=273
a04: 492 valid segments | A=453, N=39
a05: 454 valid segments | A=276, N=178
a10: 517 valid segments | A=100, N=417
a15: 510 valid segments | A=368, N=142
a20: 510 valid segments | A=315, N=195
b01: 487 valid segments | A=19, N=468
b02: 517 valid segments | A=93, N=424
b03: 441 valid segments | A=73, N=368
b04: 424 valid segments | A=10, N=414
c01: 371 valid segments | A=0, N=371
c02: 498 valid segments | A=1, N=497
c03: 454 valid segments | A=0, N=454

Total valid 1-min segments: 7211


In [8]:
# ============================================================
# CELL 4: HRV Feature Extraction (Time + Frequency + Nonlinear)
# ============================================================

def extract_hrv_features(rr):
    """Extract HRV features from an array of RR intervals (in ms)"""
    features = {}

    # --- Time Domain ---
    features['mean_rr'] = np.mean(rr)
    features['sdnn'] = np.std(rr, ddof=1)
    features['rmssd'] = np.sqrt(np.mean(np.diff(rr)**2))
    features['mean_hr'] = 60000.0 / np.mean(rr)  # bpm
    features['sdhr'] = np.std(60000.0 / rr, ddof=1)
    features['rr_range'] = np.max(rr) - np.min(rr)
    features['median_rr'] = np.median(rr)
    features['mad_rr'] = np.median(np.abs(rr - np.median(rr)))
    features['cvrr'] = features['sdnn'] / features['mean_rr'] if features['mean_rr'] != 0 else 0

    # --- Nonlinear / Statistical ---
    features['skewness'] = float(pd.Series(rr).skew())
    features['kurtosis'] = float(pd.Series(rr).kurtosis())

    # Poincare
    rr1 = rr[:-1]
    rr2 = rr[1:]
    features['sd1'] = np.std(np.subtract(rr2, rr1) / np.sqrt(2), ddof=1)
    features['sd2'] = np.std(np.add(rr2, rr1) / np.sqrt(2), ddof=1)

    # --- Frequency Domain (Welch PSD) ---
    features['beat_count'] = len(rr)

    # Interpolate RR to uniform 4 Hz for PSD
    try:
        from scipy.interpolate import interp1d
        rr_cumsum = np.cumsum(rr) / 1000.0  # seconds
        rr_cumsum = rr_cumsum - rr_cumsum[0]
        if rr_cumsum[-1] > 0 and len(rr) >= 5:
            f_interp = interp1d(rr_cumsum, rr, kind='cubic', fill_value='extrapolate')
            fs_resample = 4.0  # Hz
            t_uniform = np.arange(0, rr_cumsum[-1], 1.0/fs_resample)
            rr_uniform = f_interp(t_uniform)

            freqs, psd = welch(rr_uniform, fs=fs_resample, nperseg=min(len(rr_uniform), 256))

            vlf_mask = (freqs >= 0.003) & (freqs < 0.04)
            lf_mask = (freqs >= 0.04) & (freqs < 0.15)
            hf_mask = (freqs >= 0.15) & (freqs < 0.4)

            features['vlf_power'] = np.trapz(psd[vlf_mask], freqs[vlf_mask]) if vlf_mask.any() else 0
            features['lf_power'] = np.trapz(psd[lf_mask], freqs[lf_mask]) if lf_mask.any() else 0
            features['hf_power'] = np.trapz(psd[hf_mask], freqs[hf_mask]) if hf_mask.any() else 0
            features['total_power'] = features['vlf_power'] + features['lf_power'] + features['hf_power']
            features['lf_hf_ratio'] = features['lf_power'] / features['hf_power'] if features['hf_power'] > 0 else 0
        else:
            for k in ['vlf_power','lf_power','hf_power','total_power','lf_hf_ratio']:
                features[k] = 0
    except:
        for k in ['vlf_power','lf_power','hf_power','total_power','lf_hf_ratio']:
            features[k] = 0

    return features

# Extract features for all segments
all_features = []
all_labels = []

for rec_name, rec_data in processed.items():
    for rr_seg, label in zip(rec_data['rr_segments'], rec_data['labels']):
        feat = extract_hrv_features(rr_seg)
        all_features.append(feat)
        all_labels.append(label)

df = pd.DataFrame(all_features)
df['label'] = all_labels

print(f"Feature matrix shape: {df.shape}")
print(f"Features: {list(df.columns[:-1])}")
print(f"\nClass distribution:")
print(df['label'].value_counts().rename({1: 'Apnea', 0: 'Normal'}))
print(f"\nSample features (first 3 rows):")
df.head(3)

Feature matrix shape: (7211, 20)
Features: ['mean_rr', 'sdnn', 'rmssd', 'mean_hr', 'sdhr', 'rr_range', 'median_rr', 'mad_rr', 'cvrr', 'skewness', 'kurtosis', 'sd1', 'sd2', 'beat_count', 'vlf_power', 'lf_power', 'hf_power', 'total_power', 'lf_hf_ratio']

Class distribution:
label
Normal    4367
Apnea     2844
Name: count, dtype: int64

Sample features (first 3 rows):


,mean_rr,sdnn,rmssd,mean_hr,sdhr,rr_range,median_rr,mad_rr,cvrr,skewness,kurtosis,sd1,sd2,beat_count,vlf_power,lf_power,hf_power,total_power,lf_hf_ratio,label
0,900.000000,59.510826,41.435770,66.666667,4.457357,250.0,900.0,40.0,0.066123,0.043360,-0.516852,29.526715,79.470604,66,704.031197,1324.682258,1213.922405,3242.635860,1.091241,0
1,839.154930,46.128593,38.097619,71.500504,4.091773,240.0,840.0,30.0,0.054970,-0.452930,0.949896,27.126725,59.624081,71,198.105197,953.662895,866.566999,2018.335091,1.100507,0
2,812.837838,81.097274,42.297058,73.815461,7.475263,320.0,800.0,70.0,0.099771,0.061926,-1.251711,30.092766,110.943700,74,1346.566513,1092.387050,1284.601746,3723.555309,0.850370,0


In [9]:
# ============================================================
# CELL 5: Record-Level Train/Test Split, Clean, Scale, PCA
# ============================================================

# Record-level split to prevent data leakage
# Training: a01-a20, b01-b05, c01-c10
# Test: hold out a few records (a17-a20, c08-c10) as internal test
# (The official test set x01-x35 has no public annotations)

train_records = [f'a{i:02d}' for i in range(1,17)] + \
                [f'b{i:02d}' for i in range(1,6)] + \
                [f'c{i:02d}' for i in range(1,8)]
test_records =  [f'a{i:02d}' for i in range(17,21)] + \
                [f'c{i:02d}' for i in range(8,11)]

# Build train/test DataFrames
train_feats, train_labels = [], []
test_feats, test_labels = [], []

for rec_name, rec_data in processed.items():
    for rr_seg, label in zip(rec_data['rr_segments'], rec_data['labels']):
        feat = extract_hrv_features(rr_seg)
        if rec_name in train_records:
            train_feats.append(feat)
            train_labels.append(label)
        elif rec_name in test_records:
            test_feats.append(feat)
            test_labels.append(label)

X_train = pd.DataFrame(train_feats)
y_train = np.array(train_labels)
X_test = pd.DataFrame(test_feats)
y_test = np.array(test_labels)

print(f"Train: {X_train.shape[0]} segments | Apnea={sum(y_train==1)}, Normal={sum(y_train==0)}")
print(f"Test:  {X_test.shape[0]} segments | Apnea={sum(y_test==1)}, Normal={sum(y_test==0)}")

# Replace inf with NaN, fill NaN with training median
X_train.replace([np.inf, -np.inf], np.nan, inplace=True)
X_test.replace([np.inf, -np.inf], np.nan, inplace=True)
train_medians = X_train.median()
X_train.fillna(train_medians, inplace=True)
X_test.fillna(train_medians, inplace=True)

# StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# PCA — plot explained variance, keep 95%
pca_full = PCA()
pca_full.fit(X_train_scaled)
cumvar = np.cumsum(pca_full.explained_variance_ratio_)

plt.figure(figsize=(8, 4))
plt.plot(range(1, len(cumvar)+1), cumvar, 'bo-')
plt.axhline(y=0.95, color='r', linestyle='--', label='95% threshold')
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('PCA — Cumulative Explained Variance')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('pca_variance.png', dpi=150, bbox_inches='tight')
plt.show()

n_components_95 = np.argmax(cumvar >= 0.95) + 1
print(f"\nComponents for 95% variance: {n_components_95}")

pca = PCA(n_components=n_components_95)
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)
print(f"PCA-transformed shape — Train: {X_train_pca.shape}, Test: {X_test_pca.shape}")

Train: 6701 segments | Apnea=2529, Normal=4172
Test:  510 segments | Apnea=315, Normal=195

Components for 95% variance: 9
PCA-transformed shape — Train: (6701, 9), Test: (510, 9)


In [10]:
# ============================================================
# CELL 6: Train 7 Classifiers & Evaluate
# ============================================================

classifiers = {
    'Logistic Regression': LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42),
    'SVM-RBF': SVC(kernel='rbf', class_weight='balanced', probability=True, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
    'AdaBoost': AdaBoostClassifier(n_estimators=100, random_state=42, algorithm='SAMME'),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'Decision Tree': DecisionTreeClassifier(class_weight='balanced', random_state=42)
}

results = {}

for name, clf in classifiers.items():
    clf.fit(X_train_pca, y_train)
    y_pred = clf.predict(X_test_pca)

    if hasattr(clf, 'predict_proba'):
        y_prob = clf.predict_proba(X_test_pca)[:, 1]
    else:
        y_prob = clf.decision_function(X_test_pca)

    results[name] = {
        'y_pred': y_pred,
        'y_prob': y_prob,
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, pos_label=1),
        'recall': recall_score(y_test, y_pred, pos_label=1),
        'f1': f1_score(y_test, y_pred, pos_label=1),
    }

    fpr, tpr, _ = roc_curve(y_test, y_prob)
    results[name]['fpr'] = fpr
    results[name]['tpr'] = tpr
    results[name]['auc'] = auc(fpr, tpr)

    print(f"\n{'='*50}")
    print(f"{name}")
    print(f"{'='*50}")
    print(classification_report(y_test, y_pred, target_names=['Normal','Apnea']))

print("All classifiers trained and evaluated.")


Logistic Regression
              precision    recall  f1-score   support

      Normal       0.40      0.20      0.27       195
       Apnea       0.62      0.82      0.71       315

    accuracy                           0.58       510
   macro avg       0.51      0.51      0.49       510
weighted avg       0.54      0.58      0.54       510


SVM-RBF
              precision    recall  f1-score   support

      Normal       0.49      0.92      0.64       195
       Apnea       0.89      0.42      0.57       315

    accuracy                           0.61       510
   macro avg       0.69      0.67      0.61       510
weighted avg       0.74      0.61      0.60       510


Random Forest
              precision    recall  f1-score   support

      Normal       0.45      0.96      0.61       195
       Apnea       0.92      0.26      0.41       315

    accuracy                           0.53       510
   macro avg       0.68      0.61      0.51       510
weighted avg       0.74      

In [11]:
# ============================================================
# CELL 7: Confusion Matrices (all 7)
# ============================================================

fig, axes = plt.subplots(2, 4, figsize=(20, 9))
axes = axes.flatten()

for idx, (name, res) in enumerate(results.items()):
    cm = confusion_matrix(y_test, res['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Normal','Apnea'],
                yticklabels=['Normal','Apnea'],
                ax=axes[idx])
    axes[idx].set_title(f'{name}\nAcc={res["accuracy"]:.2f} F1={res["f1"]:.2f}')
    axes[idx].set_ylabel('Actual')
    axes[idx].set_xlabel('Predicted')

axes[-1].axis('off')  # hide 8th subplot
plt.suptitle('Confusion Matrices — All Classifiers', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

In [13]:
# ============================================================
# CELL 8: Overlapping ROC Curves
# ============================================================

plt.figure(figsize=(9, 7))

for name, res in results.items():
    plt.plot(res['fpr'], res['tpr'], linewidth=2,
             label=f"{name} (AUC={res['auc']:.3f})")

plt.plot([0,1], [0,1], 'k--', linewidth=1, label='Random')
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('ROC Curves — All Classifiers', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=9)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('roc_curves.png', dpi=150, bbox_inches='tight')
plt.show()

In [14]:
# ============================================================
# CELL 9: Summary Table sorted by F1-Score
# ============================================================

summary = pd.DataFrame({
    'Classifier': list(results.keys()),
    'Accuracy': [r['accuracy'] for r in results.values()],
    'Precision': [r['precision'] for r in results.values()],
    'Recall': [r['recall'] for r in results.values()],
    'F1-Score': [r['f1'] for r in results.values()],
    'AUC': [r['auc'] for r in results.values()]
}).sort_values('F1-Score', ascending=False).reset_index(drop=True)

print(summary.to_string(index=False))

# Save as styled image
fig, ax = plt.subplots(figsize=(10, 3))
ax.axis('off')
table = ax.table(cellText=summary.round(4).values,
                 colLabels=summary.columns,
                 cellLoc='center',
                 loc='center')
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1.2, 1.5)

# Highlight best row
for j in range(len(summary.columns)):
    table[1, j].set_facecolor('#90EE90')

plt.title('Classifier Performance Summary (sorted by F1)', fontsize=12,
          fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('summary_table.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nAll outputs saved: pca_variance.png, confusion_matrices.png, roc_curves.png, summary_table.png")

         Classifier  Accuracy  Precision   Recall  F1-Score      AUC
Logistic Regression  0.580392   0.622276 0.815873  0.706044 0.577354
            SVM-RBF  0.609804   0.891892 0.419048  0.570194 0.864290
      Decision Tree  0.584314   0.865248 0.387302  0.535088 0.644933
                KNN  0.560784   0.917431 0.317460  0.471698 0.795775
      Random Forest  0.531373   0.922222 0.263492  0.409877 0.870232
  Gradient Boosting  0.498039   0.968254 0.193651  0.322751 0.893944
           AdaBoost  0.439216   0.871795 0.107937  0.192090 0.796093

All outputs saved: pca_variance.png, confusion_matrices.png, roc_curves.png, summary_table.png


DL Model

In [15]:
# ============================================================
# BLOCK 3: Preprocessing & RR Interval Extraction
# ============================================================

def preprocess_ecg(signal, fs):
    """Bandpass + Notch + Savitzky-Golay"""
    # Bandpass 0.5-40 Hz
    b, a = butter(4, [0.5/(0.5*fs), 40.0/(0.5*fs)], btype='band')
    filtered = filtfilt(b, a, signal)
    # 50 Hz notch
    b_n, a_n = iirnotch(50.0, 30.0, fs)
    filtered = filtfilt(b_n, a_n, filtered)
    # Savitzky-Golay smoothing
    filtered = savgol_filter(filtered, window_length=7, polyorder=3)
    return filtered

def extract_rr(signal, fs):
    """R-peak detection via NeuroKit2 → RR intervals in ms"""
    try:
        _, rpeaks = nk.ecg_peaks(signal, sampling_rate=fs, method='hamilton')
        rpeak_idx = rpeaks['ECG_R_Peaks']
        rpeak_idx = rpeak_idx[~np.isnan(rpeak_idx)].astype(int)
        if len(rpeak_idx) < 3:
            return np.array([]), np.array([])
        rr = np.diff(rpeak_idx) / fs * 1000  # ms
        rr_times = rpeak_idx[1:] / fs         # seconds
        return rr, rr_times
    except:
        return np.array([]), np.array([])

# Process all records
processed = {}

for rid, rec in database.items():
    signal = rec['signal']
    fs = rec['fs']
    labels = rec['labels']

    filtered = preprocess_ecg(signal, fs)
    rr, rr_times = extract_rr(filtered, fs)

    if len(rr) < 5:
        print(f"{rid}: SKIPPED (insufficient R-peaks)")
        continue

    # Segment into 1-min windows
    n_minutes = len(labels)
    segments = []

    for i in range(n_minutes):
        t_start = i * 60
        t_end = (i + 1) * 60
        mask = (rr_times >= t_start) & (rr_times < t_end)
        rr_seg = rr[mask]

        if len(rr_seg) >= 5:
            segments.append({
                'rr': rr_seg,
                'label': 1 if labels[i] == 'A' else 0,
                'record': rid,
                'minute': i
            })

    processed[rid] = segments
    n_a = sum(1 for s in segments if s['label'] == 1)
    n_n = sum(1 for s in segments if s['label'] == 0)
    print(f"{rid}: {len(segments)} valid segments | A={n_a} N={n_n}")

total = sum(len(v) for v in processed.values())
print(f"\nTotal valid segments: {total}")

a01: 489 valid segments | A=470 N=19
a02: 528 valid segments | A=420 N=108
a03: 519 valid segments | A=246 N=273
a04: 492 valid segments | A=453 N=39
a05: 454 valid segments | A=276 N=178
a10: 517 valid segments | A=100 N=417
a15: 510 valid segments | A=368 N=142
a20: 510 valid segments | A=315 N=195
b01: 487 valid segments | A=19 N=468
b02: 517 valid segments | A=93 N=424
b03: 441 valid segments | A=73 N=368
b04: 5 valid segments | A=0 N=5
c01: 8 valid segments | A=0 N=8
c02: 6 valid segments | A=0 N=6
c03: 454 valid segments | A=0 N=454

Total valid segments: 5937


In [16]:
# ============================================================
# BLOCK 4: HRV Feature Extraction (19 features)
# ============================================================

def extract_hrv(rr):
    f = {}

    # Time domain (9)
    f['mean_rr'] = np.mean(rr)
    f['sdnn'] = np.std(rr, ddof=1)
    f['rmssd'] = np.sqrt(np.mean(np.diff(rr)**2))
    f['mean_hr'] = 60000.0 / np.mean(rr)
    f['sdhr'] = np.std(60000.0 / rr, ddof=1)
    f['rr_range'] = np.ptp(rr)
    f['median_rr'] = np.median(rr)
    f['mad_rr'] = np.median(np.abs(rr - np.median(rr)))
    f['cvrr'] = f['sdnn'] / f['mean_rr'] if f['mean_rr'] != 0 else 0

    # Nonlinear (4)
    f['skewness'] = float(skew(rr))
    f['kurtosis'] = float(kurtosis(rr))
    rr1, rr2 = rr[:-1], rr[1:]
    f['sd1'] = np.std((rr2 - rr1) / np.sqrt(2), ddof=1)
    f['sd2'] = np.std((rr2 + rr1) / np.sqrt(2), ddof=1)

    # Frequency domain (6)
    f['beat_count'] = len(rr)
    try:
        rr_cum = np.cumsum(rr) / 1000.0
        rr_cum -= rr_cum[0]
        if rr_cum[-1] > 0:
            fi = interp1d(rr_cum, rr, kind='cubic', fill_value='extrapolate')
            t_uni = np.arange(0, rr_cum[-1], 0.25)  # 4 Hz
            rr_uni = fi(t_uni)
            freqs, psd = welch(rr_uni, fs=4.0, nperseg=min(len(rr_uni), 256))

            vlf = (freqs >= 0.003) & (freqs < 0.04)
            lf  = (freqs >= 0.04)  & (freqs < 0.15)
            hf  = (freqs >= 0.15)  & (freqs < 0.4)

            f['vlf_power'] = np.trapz(psd[vlf], freqs[vlf]) if vlf.any() else 0
            f['lf_power']  = np.trapz(psd[lf], freqs[lf])   if lf.any()  else 0
            f['hf_power']  = np.trapz(psd[hf], freqs[hf])   if hf.any()  else 0
            f['total_power'] = f['vlf_power'] + f['lf_power'] + f['hf_power']
            f['lf_hf_ratio'] = f['lf_power'] / f['hf_power'] if f['hf_power'] > 0 else 0
        else:
            raise ValueError
    except:
        for k in ['vlf_power','lf_power','hf_power','total_power','lf_hf_ratio']:
            f[k] = 0

    return f

# Build feature DataFrame
rows = []
for rid, segments in processed.items():
    for seg in segments:
        feat = extract_hrv(seg['rr'])
        feat['label'] = seg['label']
        feat['record'] = seg['record']
        feat['minute'] = seg['minute']
        rows.append(feat)

df = pd.DataFrame(rows)

# Clean
feature_cols = [c for c in df.columns if c not in ['label','record','minute']]
df[feature_cols] = df[feature_cols].replace([np.inf, -np.inf], np.nan)
df[feature_cols] = df[feature_cols].fillna(df[feature_cols].median())

print(f"DataFrame shape: {df.shape}")
print(f"Features: {feature_cols}")
print(f"\nClass distribution:")
print(df['label'].value_counts().rename({1:'Apnea', 0:'Normal'}))
df.head()

DataFrame shape: (5937, 22)
Features: ['mean_rr', 'sdnn', 'rmssd', 'mean_hr', 'sdhr', 'rr_range', 'median_rr', 'mad_rr', 'cvrr', 'skewness', 'kurtosis', 'sd1', 'sd2', 'beat_count', 'vlf_power', 'lf_power', 'hf_power', 'total_power', 'lf_hf_ratio']

Class distribution:
label
Normal    3104
Apnea     2833
Name: count, dtype: int64


,mean_rr,sdnn,rmssd,mean_hr,sdhr,rr_range,median_rr,mad_rr,cvrr,skewness,...,sd2,beat_count,vlf_power,lf_power,hf_power,total_power,lf_hf_ratio,label,record,minute
0,901.363636,65.559987,58.769432,66.565809,4.933477,250.0,910.0,50.0,0.072734,-0.059421,...,82.930404,66,702.433916,1458.874126,1492.734176,3654.042218,0.977317,0,a01,0
1,839.295775,47.818823,44.336377,71.488505,4.192585,250.0,840.0,30.0,0.056975,-0.212738,...,60.262023,71,205.936528,890.681193,900.495305,1997.113027,0.989101,0,a01,1
2,812.027027,84.561394,55.319743,73.889166,7.868574,350.0,810.0,80.0,0.104136,0.029745,...,112.797973,74,1369.152265,1103.134021,1379.278633,3851.564920,0.799791,0,a01,2
3,749.125000,62.301711,33.715113,80.093442,7.183352,260.0,760.0,40.0,0.083166,-0.626412,...,84.832825,80,1054.561595,964.432852,446.776014,2465.770461,2.158650,0,a01,3
4,794.736842,47.510664,34.717911,75.496689,4.492731,230.0,790.0,20.0,0.059782,0.328671,...,62.740995,76,75.580104,1415.602314,416.885593,1908.068012,3.395661,0,a01,4


In [17]:
# ============================================================
# BLOCK 5: Save HRV Features to CSV
# ============================================================

os.makedirs('output', exist_ok=True)

df.to_csv('output/hrv_features.csv', index=False)
print(f"Saved: output/hrv_features.csv ({df.shape[0]} rows, {df.shape[1]} cols)")

# Also save a clean version without metadata for direct model input
df_model = df.drop(columns=['record','minute'])
df_model.to_csv('output/hrv_features_model.csv', index=False)
print(f"Saved: output/hrv_features_model.csv (features + label only)")

Saved: output/hrv_features.csv (5937 rows, 22 cols)
Saved: output/hrv_features_model.csv (features + label only)


In [18]:
# ============================================================
# BLOCK 6: Generate CWT Scalogram Images from RR Intervals
# ============================================================

IMG_SIZE = 128
os.makedirs('output/images/apnea', exist_ok=True)
os.makedirs('output/images/normal', exist_ok=True)

def rr_to_scalogram(rr, img_size=IMG_SIZE):
    """Convert RR interval array to CWT scalogram image (grayscale)"""
    # Interpolate to fixed length
    x_old = np.linspace(0, 1, len(rr))
    x_new = np.linspace(0, 1, img_size)
    rr_interp = np.interp(x_new, x_old, rr)

    # Normalize
    rr_interp = (rr_interp - np.mean(rr_interp)) / (np.std(rr_interp) + 1e-8)

    # CWT with Morlet wavelet
    scales = np.arange(1, img_size + 1)
    coeffs, _ = pywt.cwt(rr_interp, scales, 'morl')

    # Normalize coefficients to 0-255
    coeffs = np.abs(coeffs)
    coeffs = (coeffs - coeffs.min()) / (coeffs.max() - coeffs.min() + 1e-8)
    return coeffs

# Generate and save images
img_count = {'apnea': 0, 'normal': 0}
image_index = []  # track image paths + labels

for rid, segments in processed.items():
    for seg in segments:
        rr = seg['rr']
        label = seg['label']
        minute = seg['minute']
        cls = 'apnea' if label == 1 else 'normal'

        scalogram = rr_to_scalogram(rr)

        fname = f"{rid}_min{minute:04d}.png"
        fpath = f"output/images/{cls}/{fname}"

        plt.imsave(fpath, scalogram, cmap='jet')
        image_index.append({'filename': fname, 'path': fpath, 'label': label, 'record': rid})
        img_count[cls] += 1

# Save image index
img_df = pd.DataFrame(image_index)
img_df.to_csv('output/image_index.csv', index=False)

print(f"Scalogram images generated:")
print(f"  Apnea:  {img_count['apnea']}")
print(f"  Normal: {img_count['normal']}")
print(f"  Total:  {sum(img_count.values())}")
print(f"  Size:   {IMG_SIZE}x{IMG_SIZE} pixels")
print(f"  Saved to: output/images/apnea/ and output/images/normal/")

# Show sample images
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
samples = img_df.groupby('label').apply(lambda x: x.sample(2, random_state=42)).reset_index(drop=True)
for idx, (_, row) in enumerate(samples.iterrows()):
    img = plt.imread(row['path'])
    axes[idx].imshow(img)
    axes[idx].set_title(f"{'Apnea' if row['label']==1 else 'Normal'}\n{row['filename']}", fontsize=9)
    axes[idx].axis('off')
plt.suptitle('Sample CWT Scalograms', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('output/sample_scalograms.png', dpi=150)
plt.show()

Scalogram images generated:
  Apnea:  2833
  Normal: 3104
  Total:  5937
  Size:   128x128 pixels
  Saved to: output/images/apnea/ and output/images/normal/


In [35]:
# ============================================================
# BLOCK 7 v3: Dual-Input Dataset (Raw RR + HRV Features)
# ============================================================

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

# --- Build unified dataset from processed data ---
RR_FIXED_LEN = 120

all_samples = []
for rid, segments in processed.items():
    for seg in segments:
        rr = seg['rr']
        feat = extract_hrv(rr)

        if len(rr) >= RR_FIXED_LEN:
            rr_fixed = rr[:RR_FIXED_LEN]
        else:
            rr_fixed = np.pad(rr, (0, RR_FIXED_LEN - len(rr)), mode='constant', constant_values=0)

        all_samples.append({
            'rr_raw': rr_fixed,
            'hrv_features': np.array([feat[k] for k in sorted(feat.keys())], dtype=np.float32),
            'feature_names': sorted(feat.keys()),
            'label': seg['label'],
            'record': seg['record']
        })

print(f"Total samples: {len(all_samples)}")
print(f"RR fixed length: {RR_FIXED_LEN}")
print(f"HRV features: {len(all_samples[0]['hrv_features'])}")

# --- Record-level split ---
sample_df = pd.DataFrame([{'record': s['record'], 'label': s['label']} for s in all_samples])
all_records = sample_df['record'].unique()
rec_stats = sample_df.groupby('record')['label'].mean().reset_index()
rec_stats.columns = ['record', 'apnea_ratio']
rec_stats['bin'] = (rec_stats['apnea_ratio'] > 0.5).astype(int)

train_recs, temp_recs = train_test_split(
    rec_stats['record'].values, test_size=0.3,
    stratify=rec_stats['bin'].values, random_state=42)
temp_stats = rec_stats[rec_stats['record'].isin(temp_recs)]
val_recs, test_recs = train_test_split(
    temp_stats['record'].values, test_size=0.5,
    stratify=temp_stats['bin'].values, random_state=42)

train_idx = [i for i, s in enumerate(all_samples) if s['record'] in train_recs]
val_idx   = [i for i, s in enumerate(all_samples) if s['record'] in val_recs]
test_idx  = [i for i, s in enumerate(all_samples) if s['record'] in test_recs]

print(f"\nTrain: {len(train_idx)} | Val: {len(val_idx)} | Test: {len(test_idx)}")
print(f"Train recs: {sorted(train_recs)}")
print(f"Val recs:   {sorted(val_recs)}")
print(f"Test recs:  {sorted(test_recs)}")

# --- Normalize RR and HRV features using training stats ---
train_rr = np.array([all_samples[i]['rr_raw'] for i in train_idx])
train_hrv = np.array([all_samples[i]['hrv_features'] for i in train_idx])

rr_mean, rr_std = train_rr[train_rr != 0].mean(), train_rr[train_rr != 0].std()
hrv_scaler = StandardScaler()
hrv_scaler.fit(train_hrv)

for i, s in enumerate(all_samples):
    rr = s['rr_raw'].copy()
    mask = rr != 0
    rr[mask] = (rr[mask] - rr_mean) / (rr_std + 1e-8)
    all_samples[i]['rr_norm'] = rr
    all_samples[i]['hrv_norm'] = hrv_scaler.transform(s['hrv_features'].reshape(1, -1)).flatten()

for s in all_samples:
    s['hrv_norm'] = np.nan_to_num(s['hrv_norm'], nan=0, posinf=0, neginf=0)

# --- Dataset ---
class DualInputDataset(Dataset):
    def __init__(self, indices, samples, augment=False):
        self.indices = indices
        self.samples = samples
        self.augment = augment

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        s = self.samples[self.indices[idx]]

        rr = torch.tensor(s['rr_norm'], dtype=torch.float32).unsqueeze(0)
        hrv = torch.tensor(s['hrv_norm'], dtype=torch.float32)
        label = torch.tensor(s['label'], dtype=torch.long)

        if self.augment:
            noise = torch.randn_like(rr) * 0.05
            scale = 1.0 + (torch.rand(1) - 0.5) * 0.1
            rr = rr * scale + noise

        return rr, hrv, label

# --- WeightedRandomSampler ---
train_labels = np.array([all_samples[i]['label'] for i in train_idx])
class_counts = np.bincount(train_labels)
sample_weights = np.array([1.0/class_counts[l] for l in train_labels])
sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

BATCH_SIZE = 64

train_loader = DataLoader(DualInputDataset(train_idx, all_samples, augment=True),
                           batch_size=BATCH_SIZE, sampler=sampler, num_workers=2,
                           drop_last=True)
val_loader = DataLoader(DualInputDataset(val_idx, all_samples, augment=False),
                         batch_size=BATCH_SIZE, shuffle=False, num_workers=2,
                         drop_last=True)
test_loader = DataLoader(DualInputDataset(test_idx, all_samples, augment=False),
                          batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# Verify
rr_b, hrv_b, lab_b = next(iter(train_loader))
print(f"\nRR batch:  {rr_b.shape}")
print(f"HRV batch: {hrv_b.shape}")
print(f"Labels:    {dict(zip(*np.unique(lab_b.numpy(), return_counts=True)))}")
print("WeightedRandomSampler active — each batch ~50/50 apnea/normal")

Device: cuda
Total samples: 5937
RR fixed length: 120
HRV features: 19

Train: 3969 | Val: 1027 | Test: 941
Train recs: ['a01', 'a02', 'a03', 'a05', 'a10', 'a15', 'b01', 'b04', 'c02', 'c03']
Val recs:   ['a20', 'b02']
Test recs:  ['a04', 'b03', 'c01']

RR batch:  torch.Size([64, 1, 120])
HRV batch: torch.Size([64, 19])
Labels:    {np.int64(0): np.int64(30), np.int64(1): np.int64(34)}
WeightedRandomSampler active — each batch ~50/50 apnea/normal


In [36]:
# ============================================================
# BLOCK 8 v3: Dual-Input Model (1D-CNN on RR + Dense on HRV)
# ============================================================

class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha

    def forward(self, inputs, targets):
        ce = nn.functional.cross_entropy(inputs, targets, weight=self.alpha, reduction='none')
        pt = torch.exp(-ce)
        return (((1 - pt) ** self.gamma) * ce).mean()


class ApneaDualNet(nn.Module):
    def __init__(self, rr_len=120, hrv_dim=19):
        super().__init__()

        # --- Branch 1: 1D-CNN on raw RR intervals ---
        self.rr_branch = nn.Sequential(
            # Block 1
            nn.Conv1d(1, 64, kernel_size=7, padding=3),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
            nn.Conv1d(64, 64, kernel_size=5, padding=2),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(2),
            nn.Dropout(0.3),

            # Block 2
            nn.Conv1d(64, 128, kernel_size=5, padding=2),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Conv1d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(2),
            nn.Dropout(0.3),

            # Block 3
            nn.Conv1d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool1d(1),  # Global Average Pooling
        )

        # --- Branch 2: Dense on HRV features ---
        self.hrv_branch = nn.Sequential(
            nn.Linear(hrv_dim, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(64, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
        )

        # --- Fusion ---
        self.fusion = nn.Sequential(
            nn.Linear(256 + 64, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(128, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(64, 2)
        )

    def forward(self, rr, hrv):
        # RR branch: (B, 1, 120) → (B, 256)
        rr_out = self.rr_branch(rr).squeeze(-1)

        # HRV branch: (B, 19) → (B, 64)
        hrv_out = self.hrv_branch(hrv)

        # Concatenate and classify
        combined = torch.cat([rr_out, hrv_out], dim=1)
        return self.fusion(combined)


hrv_dim = hrv_b.shape[1]
model = ApneaDualNet(rr_len=RR_FIXED_LEN, hrv_dim=hrv_dim).to(device)

# Focal loss
alpha = torch.tensor([1.0/class_counts[0], 1.0/class_counts[1]],
                      dtype=torch.float32).to(device)
alpha = alpha / alpha.sum() * 2

criterion = FocalLoss(alpha=alpha, gamma=2.0)
optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-3)
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2)

print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total_params:,}")
print(f"RR input: (1, {RR_FIXED_LEN})")
print(f"HRV input: ({hrv_dim},)")

ApneaDualNet(
  (rr_branch): Sequential(
    (0): Conv1d(1, 64, kernel_size=(7,), stride=(1,), padding=(3,))
    (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): Conv1d(64, 64, kernel_size=(5,), stride=(1,), padding=(2,))
    (4): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (5): ReLU(inplace=True)
    (6): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (7): Dropout(p=0.3, inplace=False)
    (8): Conv1d(64, 128, kernel_size=(5,), stride=(1,), padding=(2,))
    (9): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU(inplace=True)
    (11): Conv1d(128, 128, kernel_size=(3,), stride=(1,), padding=(1,))
    (12): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (13): ReLU(inplace=True)
    (14): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)

In [37]:
# ============================================================
# BLOCK 9 v3: Training with F1-based Early Stopping
# ============================================================

from sklearn.metrics import f1_score as compute_f1

EPOCHS = 100
PATIENCE = 15

history = {'train_loss':[], 'val_loss':[], 'train_acc':[], 'val_acc':[],
           'val_f1':[], 'lr':[]}
best_val_f1 = 0.0
patience_counter = 0
best_model_state = None

for epoch in range(EPOCHS):
    # --- Train ---
    model.train()
    train_loss, train_correct, train_total = 0, 0, 0

    for rr, hrv, labels in train_loader:
        rr, hrv, labels = rr.to(device), hrv.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(rr, hrv)
        loss = criterion(outputs, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        train_loss += loss.item() * rr.size(0)
        train_correct += (outputs.argmax(1) == labels).sum().item()
        train_total += rr.size(0)

    scheduler.step()
    train_loss /= train_total
    train_acc = train_correct / train_total

    # --- Validate ---
    model.eval()
    val_loss, val_correct, val_total = 0, 0, 0
    val_preds, val_true = [], []

    with torch.no_grad():
        for rr, hrv, labels in val_loader:
            rr, hrv, labels = rr.to(device), hrv.to(device), labels.to(device)
            outputs = model(rr, hrv)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * rr.size(0)
            val_correct += (outputs.argmax(1) == labels).sum().item()
            val_total += rr.size(0)
            val_preds.extend(outputs.argmax(1).cpu().numpy())
            val_true.extend(labels.cpu().numpy())

    val_loss /= val_total
    val_acc = val_correct / val_total
    val_f1 = compute_f1(val_true, val_preds, pos_label=1)
    current_lr = optimizer.param_groups[0]['lr']

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)
    history['val_f1'].append(val_f1)
    history['lr'].append(current_lr)

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        patience_counter = 0
        best_model_state = {k: v.clone() for k, v in model.state_dict().items()}
        marker = '★ BEST'
    else:
        patience_counter += 1
        marker = ''

    if (epoch+1) % 5 == 0 or marker:
        print(f"Epoch {epoch+1:3d}/{EPOCHS} | "
              f"TrL={train_loss:.4f} TrA={train_acc:.4f} | "
              f"VaL={val_loss:.4f} VaA={val_acc:.4f} VaF1={val_f1:.4f} | "
              f"LR={current_lr:.6f} {marker}")

    if patience_counter >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch+1}")
        break

model.load_state_dict(best_model_state)
print(f"\nBest model restored (val_f1={best_val_f1:.4f})")

Epoch   1/100 | TrL=0.1481 TrA=0.6855 | VaL=0.2005 VaA=0.6377 VaF1=0.2624 | LR=0.000976 ★ BEST
Epoch   2/100 | TrL=0.1201 TrA=0.7757 | VaL=0.1481 VaA=0.6787 VaF1=0.4598 | LR=0.000905 ★ BEST
Epoch   5/100 | TrL=0.1066 TrA=0.8082 | VaL=0.1538 VaA=0.6426 VaF1=0.2651 | LR=0.000500 
Epoch   9/100 | TrL=0.1056 TrA=0.8173 | VaL=0.1400 VaA=0.7129 VaF1=0.6101 | LR=0.000024 ★ BEST
Epoch  10/100 | TrL=0.0998 TrA=0.8317 | VaL=0.1568 VaA=0.6328 VaF1=0.2068 | LR=0.001000 
Epoch  15/100 | TrL=0.1005 TrA=0.8191 | VaL=0.1460 VaA=0.7178 VaF1=0.6232 | LR=0.000854 ★ BEST
Epoch  16/100 | TrL=0.0991 TrA=0.8254 | VaL=0.1363 VaA=0.7451 VaF1=0.6897 | LR=0.000794 ★ BEST
Epoch  20/100 | TrL=0.0930 TrA=0.8387 | VaL=0.1785 VaA=0.5996 VaF1=0.0000 | LR=0.000500 
Epoch  25/100 | TrL=0.0915 TrA=0.8450 | VaL=0.1920 VaA=0.6016 VaF1=0.0192 | LR=0.000146 
Epoch  30/100 | TrL=0.0885 TrA=0.8392 | VaL=0.1279 VaA=0.7568 VaF1=0.6937 | LR=0.001000 ★ BEST
Epoch  35/100 | TrL=0.0929 TrA=0.8448 | VaL=0.1664 VaA=0.5996 VaF1=0.0049 

In [38]:
# ============================================================
# BLOCK 10 v3: Training Curves
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(history['train_loss'], label='Train', lw=2)
axes[0].plot(history['val_loss'], label='Val', lw=2)
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(history['train_acc'], label='Train', lw=2)
axes[1].plot(history['val_acc'], label='Val', lw=2)
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].set_title('Accuracy'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

axes[2].plot(history['val_f1'], label='Val F1', lw=2, color='green')
axes[2].axhline(y=best_val_f1, color='r', linestyle='--', alpha=0.5, label=f'Best={best_val_f1:.3f}')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('F1')
axes[2].set_title('Apnea F1'); axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.suptitle('Dual-Input Model Training', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('output/training_curves_v3.png', dpi=150)
plt.show()

In [39]:
# ============================================================
# BLOCK 11 v3: Test Evaluation + Threshold Tuning
# ============================================================

from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_curve, auc, precision_recall_curve,
                             accuracy_score, f1_score, precision_score, recall_score)
import seaborn as sns

model.eval()
all_preds, all_labels, all_probs = [], [], []

with torch.no_grad():
    for rr, hrv, labels in test_loader:
        rr, hrv = rr.to(device), hrv.to(device)
        outputs = model(rr, hrv)
        probs = torch.softmax(outputs, dim=1)[:, 1]
        all_preds.extend(outputs.argmax(1).cpu().numpy())
        all_labels.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy())

all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
all_probs = np.array(all_probs)

# --- Default threshold ---
print("=" * 55)
print("TEST RESULTS — Default Threshold (0.5)")
print("=" * 55)
print(classification_report(all_labels, all_preds, target_names=['Normal','Apnea']))

# --- Sweep thresholds ---
thresholds = np.arange(0.20, 0.80, 0.01)
f1_scores = [f1_score(all_labels, (all_probs >= t).astype(int), pos_label=1) for t in thresholds]
acc_scores = [accuracy_score(all_labels, (all_probs >= t).astype(int)) for t in thresholds]

# Find threshold that maximizes (F1 + Accuracy) / 2 — balanced tradeoff
combined = [(f + a) / 2 for f, a in zip(f1_scores, acc_scores)]
best_thresh = thresholds[np.argmax(combined)]

opt_preds = (all_probs >= best_thresh).astype(int)

print(f"\nOptimal threshold: {best_thresh:.2f}")
print("\n" + "=" * 55)
print(f"TEST RESULTS — Optimal Threshold ({best_thresh:.2f})")
print("=" * 55)
print(classification_report(all_labels, opt_preds, target_names=['Normal','Apnea']))

# --- Plots ---
fig, axes = plt.subplots(2, 3, figsize=(20, 12))

# 1. Threshold vs F1 & Accuracy
axes[0,0].plot(thresholds, f1_scores, 'b-', lw=2, label='F1(Apnea)')
axes[0,0].plot(thresholds, acc_scores, 'orange', lw=2, label='Accuracy')
axes[0,0].plot(thresholds, combined, 'g--', lw=2, label='Combined')
axes[0,0].axvline(x=best_thresh, color='r', linestyle='--', label=f'Best={best_thresh:.2f}')
axes[0,0].set_xlabel('Threshold'); axes[0,0].set_ylabel('Score')
axes[0,0].set_title('Threshold Tuning'); axes[0,0].legend(fontsize=8); axes[0,0].grid(True, alpha=0.3)

# 2. Confusion Matrix (default 0.5)
cm1 = confusion_matrix(all_labels, all_preds)
sns.heatmap(cm1, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal','Apnea'], yticklabels=['Normal','Apnea'], ax=axes[0,1])
axes[0,1].set_title(f'CM — Thresh=0.50\nAcc={accuracy_score(all_labels, all_preds):.3f}')
axes[0,1].set_ylabel('Actual'); axes[0,1].set_xlabel('Predicted')

# 3. Confusion Matrix (optimal)
cm2 = confusion_matrix(all_labels, opt_preds)
sns.heatmap(cm2, annot=True, fmt='d', cmap='Greens',
            xticklabels=['Normal','Apnea'], yticklabels=['Normal','Apnea'], ax=axes[0,2])
axes[0,2].set_title(f'CM — Thresh={best_thresh:.2f}\nAcc={accuracy_score(all_labels, opt_preds):.3f}')
axes[0,2].set_ylabel('Actual'); axes[0,2].set_xlabel('Predicted')

# 4. ROC Curve
fpr, tpr, _ = roc_curve(all_labels, all_probs)
roc_auc = auc(fpr, tpr)
axes[1,0].plot(fpr, tpr, 'b-', lw=2, label=f'DualNet (AUC={roc_auc:.3f})')
axes[1,0].plot([0,1], [0,1], 'k--', lw=1)
axes[1,0].set_xlabel('FPR'); axes[1,0].set_ylabel('TPR')
axes[1,0].set_title('ROC Curve'); axes[1,0].legend(); axes[1,0].grid(True, alpha=0.3)

# 5. Precision-Recall Curve
prec, rec, _ = precision_recall_curve(all_labels, all_probs)
pr_auc = auc(rec, prec)
axes[1,1].plot(rec, prec, 'g-', lw=2, label=f'PR AUC={pr_auc:.3f}')
baseline = sum(all_labels)/len(all_labels)
axes[1,1].axhline(y=baseline, color='r', linestyle='--', alpha=0.5, label=f'Baseline={baseline:.2f}')
axes[1,1].set_xlabel('Recall'); axes[1,1].set_ylabel('Precision')
axes[1,1].set_title('PR Curve'); axes[1,1].legend(); axes[1,1].grid(True, alpha=0.3)

# 6. Probability Distribution
axes[1,2].hist(all_probs[np.array(all_labels)==0], bins=30, alpha=0.6, label='Normal', color='blue')
axes[1,2].hist(all_probs[np.array(all_labels)==1], bins=30, alpha=0.6, label='Apnea', color='red')
axes[1,2].axvline(x=best_thresh, color='green', linestyle='--', lw=2, label=f'Thresh={best_thresh:.2f}')
axes[1,2].set_xlabel('Predicted Probability (Apnea)'); axes[1,2].set_ylabel('Count')
axes[1,2].set_title('Probability Distribution'); axes[1,2].legend(); axes[1,2].grid(True, alpha=0.3)

plt.suptitle('Dual-Input Model — Full Evaluation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('output/test_evaluation_v3.png', dpi=150)
plt.show()

# --- Final comparison table ---
print("\n" + "=" * 60)
print("FINAL COMPARISON")
print("=" * 60)
compare = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision(Apnea)', 'Recall(Apnea)',
               'F1(Apnea)', 'Precision(Normal)', 'Recall(Normal)',
               'F1(Normal)', 'AUC-ROC', 'PR-AUC'],
    'Thresh=0.50': [
        accuracy_score(all_labels, all_preds),
        precision_score(all_labels, all_preds, pos_label=1),
        recall_score(all_labels, all_preds, pos_label=1),
        f1_score(all_labels, all_preds, pos_label=1),
        precision_score(all_labels, all_preds, pos_label=0),
        recall_score(all_labels, all_preds, pos_label=0),
        f1_score(all_labels, all_preds, pos_label=0),
        roc_auc, pr_auc
    ],
    f'Thresh={best_thresh:.2f}': [
        accuracy_score(all_labels, opt_preds),
        precision_score(all_labels, opt_preds, pos_label=1),
        recall_score(all_labels, opt_preds, pos_label=1),
        f1_score(all_labels, opt_preds, pos_label=1),
        precision_score(all_labels, opt_preds, pos_label=0),
        recall_score(all_labels, opt_preds, pos_label=0),
        f1_score(all_labels, opt_preds, pos_label=0),
        roc_auc, pr_auc
    ]
})
print(compare.round(4).to_string(index=False))

TEST RESULTS — Default Threshold (0.5)
              precision    recall  f1-score   support

      Normal       0.92      0.75      0.83       415
       Apnea       0.83      0.95      0.88       526

    accuracy                           0.86       941
   macro avg       0.87      0.85      0.85       941
weighted avg       0.87      0.86      0.86       941


Optimal threshold: 0.49

TEST RESULTS — Optimal Threshold (0.49)
              precision    recall  f1-score   support

      Normal       0.95      0.73      0.83       415
       Apnea       0.82      0.97      0.89       526

    accuracy                           0.87       941
   macro avg       0.88      0.85      0.86       941
weighted avg       0.88      0.87      0.86       941


FINAL COMPARISON
           Metric  Thresh=0.50  Thresh=0.49
         Accuracy       0.8608       0.8650
 Precision(Apnea)       0.8264       0.8223
    Recall(Apnea)       0.9506       0.9677
        F1(Apnea)       0.8842       0.8891
Pre